In [1]:
import sys
sys.version

'3.11.9 | packaged by conda-forge | (main, Apr 19 2024, 18:36:13) [GCC 12.3.0]'

In [2]:
import os
import IPython
import librosa
import subprocess
import json
from math import sqrt
import wave
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pylab
from statistics import mean, stdev
from scipy import signal
from scipy.io import wavfile
from scipy.fftpack import fft
from scipy.fft import fftshift

### Sample audio file (no preprocessing)

In [3]:
IPython.display.Audio("PSUH/pre/01pre.wav")
#IPython.display.Audio("torgo_data/dysarthria_male/M03_Session2_0321.wav")

#### Visualize waveform spectrogram

In [4]:
def demo_wav(fpath):
    fs, x = wavfile.read(fpath)
    print('fs', fs)
    print('x.shape', x.shape)
    
    nperseg = 1025
    noverlap = nperseg - 1
    f, t, Sxx = signal.spectrogram(x, fs,
                                   nperseg=nperseg,
                                   noverlap=noverlap,
                                   window='hann')
    print('f.shape', f.shape)
    print('t.shape', t.shape)
    print('Sxx.shape', Sxx.shape)
    plt.pcolormesh(1000*t, f/1000, 10*np.log10(Sxx/Sxx.max()),
                   vmin=-120, vmax=0, cmap='inferno')
    plt.ylabel('Frequency [kHz]')
    plt.xlabel('Time [ms]')
    plt.colorbar()

In [ ]:
demo_wav("PSUH/pre/01pre.wav")

# Apply signal preprocessing to audio files

### Step 1) Apply audio denoising to all sample files (remove background noise)

In [6]:
import noisereduce as nr
import warnings

with warnings.catch_warnings(): # create a context for ignoring warnings
    warnings.filterwarnings("ignore", category=wavfile.WavFileWarning)

/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
# perform noise reduction on speech audio files and save noise-reduced files in new directory
def denoise(target, SA_dir):
    files = os.listdir(SA_dir)
    for f in files:
        f_loc = SA_dir+f                                          # specify original file location
        fs, x = wavfile.read(f_loc)
        reduced_noise = nr.reduce_noise(y=x, sr=fs, n_fft=512)
        new_fname = os.path.join(target, f[:-4] + "_nr" + f[-4:]) # specify new file location target name
        wavfile.write(new_fname, fs, reduced_noise)               # write to new file location

In [ ]:
# Unaltered recordings
pre = "PSUH/pre/"
post = "PSUH/post/"
dc = "PSUH/dc/"
# New location of noise-reduced recordings
NR_pre_files = "PSUH/NR/pre/"
NR_post_files = "PSUH/NR/post/"
NR_dc_files = "PSUH/NR/dc/"
# Apply reduce_noise function on audio recordings
denoise(NR_pre_files, pre)
denoise(NR_post_files, post)
denoise(NR_dc_files, dc)

In [ ]:
# Show noise-reduced file
demo_wav("PSUH/NR/pre/01pre_nr.wav")

In [10]:
IPython.display.Audio("PSUH/NR/pre/01pre_nr.wav")

### Step 2) Audio segmentation (separate samples into individual files)

In [11]:
from vosk import Model, KaldiRecognizer

In [ ]:
# Initialize vosk English language
model = Model(lang="en-us")
# Try recognition on one sample
wf = wave.open('PSUH/NR/pre/01pre_nr.wav', 'rb')
#rec = KaldiRecognizer(model, wf.getframerate())
rec = KaldiRecognizer(model, wf.getframerate(), '["pennsylvania"]')
rec.SetWords(True)
rec.SetPartialWords(True)

while True:
    data = wf.readframes(4000)
    if len(data) == 0:
        break
    if rec.AcceptWaveform(data):
        print(rec.Result())
    else:
        print(rec.PartialResult())

print(rec.FinalResult())

In [13]:
os.chdir('PSUH/NR/')
os.getcwd()

'/home/patrick/ssr/PSUH/NR'

In [14]:
from pydub import AudioSegment

# List all noise-reduced audio files
pre_audio_files = [f for f in os.listdir('pre/') if f.endswith('_nr.wav') and os.path.isfile(os.path.join('pre/', f))]
post_audio_files = [f for f in os.listdir('post/') if f.endswith('_nr.wav') and os.path.isfile(os.path.join('post/', f))]
dc_audio_files = [f for f in os.listdir('dc/') if f.endswith('_nr.wav') and os.path.isfile(os.path.join('dc/', f))]

/opt/conda/lib/python3.11/site-packages/pydub/utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


In [15]:
# Loop through all audio files and segment them into new sample audio files
def file_segment(audio_files, dirpath, newpath):
    for file in audio_files:
        audio_filename = dirpath + file
        segment_fname = newpath + file
        wf = wave.open(audio_filename, "rb")
        
        # Check the audio format
        if wf.getnchannels() != 1 or wf.getsampwidth() != 2 or wf.getcomptype() != "NONE":
            print("Audio file must be WAV format mono PCM.")
            exit(1)
        
        # Create a recognizer and enable word timestamps
        rec = KaldiRecognizer(model, wf.getframerate(), '["pennsylvania"]')
        rec.SetWords(True)
        
        # Recognize speech and store the results in a list
        results = []
        while True:
            data = wf.readframes(4000)
            if len(data) == 0:
                break
            if rec.AcceptWaveform(data):
                results.append(json.loads(rec.Result()))
        results.append(json.loads(rec.FinalResult()))
        
        # Close the audio file
        wf.close()
        
        # Create a dataframe to store the word and its start and end times
        df = pd.DataFrame(columns=["word", "start", "end"])
        
        # Loop through the results and extract the word and its times
        for result in results:
            if "result" in result:
                for word in result["result"]:
                    # Check if the word is "test"
                    if word["word"] == "pennsylvania":
                        # Create a temporary dataframe with the word and its times
                        temp_df = pd.DataFrame({"word": [word["word"]], "start": [word["start"]], "end": [word["end"]]})
                        # Concatenate the temporary dataframe with the main dataframe
                        df = pd.concat([df, temp_df], ignore_index=True)
        
        # Segment each audio file and write into new .WAV file
        for index, row in df.iterrows():
            sample_num = index+1
            sample_num = str(sample_num)
            audio = AudioSegment.from_wav(audio_filename)
            audio_chunk=audio[row["start"]*1000:row["end"]*1000]
            audio_chunk.export(os.path.join(segment_fname[:-4]+ "_" + sample_num + ".wav"), format="wav")

In [ ]:
file_segment(pre_audio_files, 'pre/', 'pre/segment/')
file_segment(post_audio_files, 'post/', 'post/segment/')
file_segment(dc_audio_files, 'dc/', 'dc/segment/')

In [ ]:
# Show noise-reduced utterance file
demo_wav("pre/segment/01pre_nr_1.wav")

In [ ]:
demo_wav("pre/segment/01pre_nr_2.wav")

In [ ]:
demo_wav("pre/segment/01pre_nr_3.wav")

### Step 3) Describe dataset

#### Count number of utterance files for each patient and condition

In [17]:
from collections import defaultdict


def count_unique_files(directory):
    unique_files = defaultdict(set)

    # Iterate through files in the directory
    for filename in os.listdir(directory):
        if filename.endswith('.wav'):
            # Extract the ID from the filename (assuming the ID is the first two characters)
            file_id = filename[:2]
            unique_files[file_id].add(filename)

    # Count the number of unique files for each ID
    unique_file_counts = {file_id: len(files) for file_id, files in unique_files.items()}

    return unique_file_counts

In [18]:
pre_utterances = 'pre/segment/'
post_utterances = 'post/segment/'
dc_utterances = 'dc/segment/'
pre_file_counts = count_unique_files(pre_utterances)
post_file_counts = count_unique_files(post_utterances)
dc_file_counts = count_unique_files(dc_utterances)

In [20]:
# Create dataframe for each dictionary of patient utterance counts
df1 = pd.DataFrame(list(pre_file_counts.items()), columns=['ID', 'Utterances_pre'])
df2 = pd.DataFrame(list(post_file_counts.items()), columns=['ID', 'Utterances_post'])
df3 = pd.DataFrame(list(dc_file_counts.items()), columns=['ID', 'Utterances_dc'])
# Merge all dataframes by patient 'ID' column
df_combined = pd.merge(df1, df2, on='ID', how='outer')
df_combined = pd.merge(df_combined, df3, on='ID', how='outer')

In [21]:
# Fill NaN values with 0
df_combined = df_combined.fillna(0)

In [22]:
df_combined.describe()

,Utterances_pre,Utterances_post,Utterances_dc
count,48.000000,48.000000,48.000000
mean,9.583333,2.791667,2.833333
std,1.088300,0.770696,0.780980
min,5.000000,0.000000,0.000000
25%,10.000000,3.000000,3.000000
50%,10.000000,3.000000,3.000000
75%,10.000000,3.000000,3.000000
max,10.000000,4.000000,4.000000


In [23]:
df_combined.sum()

ID                 0102030405070809101112131415161718192021222324...
Utterances_pre                                                   460
Utterances_post                                                134.0
Utterances_dc                                                  136.0
dtype: object

#### Describe common spectrogram features of all PSUH utterances

In [ ]:
pre_file_list = [os.path.join(pre_utterances, file) for file in os.listdir(pre_utterances)] 
post_file_list = [os.path.join(post_utterances, file) for file in os.listdir(post_utterances)]
dc_file_list = [os.path.join(dc_utterances, file) for file in os.listdir(dc_utterances)]
pre_file_list[0]

In [ ]:
# Function to extract MFCCs from a spectrogram
def extract_mfcc(file_path):
    y, sr = librosa.load(file_path)
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    return mfccs

In [ ]:
# Extract MFCCs for all spectrograms in a directory
def summarize_spectrograms(file_paths):
    mfccs_list = [extract_mfcc(file) for file in file_paths]
    
    # Stack MFCCs to create a feature matrix
    feature_matrix = np.hstack(mfccs_list)
    
    # Calculate the mean MFCCs across all spectrograms
    mean_mfccs = np.mean(feature_matrix, axis=1)
    
    print("Mean MFCCs across all spectrograms:", mean_mfccs)

In [ ]:
summarize_spectrograms(pre_file_list)

In [ ]:
summarize_spectrograms(post_file_list)

In [ ]:
summarize_spectrograms(dc_file_list)

In [ ]:
def read_wave(filename):
    with wave.open(filename, 'r') as wf:
        n_channels = wf.getnchannels()
        sampwidth = wf.getsampwidth()
        framerate = wf.getframerate()
        n_frames = wf.getnframes()
        audio = wf.readframes(n_frames)
        waveform = np.frombuffer(audio, dtype=np.int16)
        waveform = waveform / (2**(sampwidth * 8 - 1))  # Normalize
        return waveform, framerate


# File paths
file1 = "pre/segment/01pre_nr_1.wav"
file2 = "post/segment/01post_nr_1.wav"
file3 = "dc/segment/01dc_nr_1.wav"

# Read wave files
waveform1, fs1 = read_wave(file1)
waveform2, fs2 = read_wave(file2)
waveform3, fs3 = read_wave(file3)

# Create subplots
fig, axs = plt.subplots(3, 2, figsize=(12, 8))

# Plot waveforms
t1 = np.linspace(0, len(waveform1) / fs1, num=len(waveform1))
axs[0, 0].plot(t1, waveform1)
axs[0, 0].set_title(file1)

t2 = np.linspace(0, len(waveform2) / fs2, num=len(waveform2))
axs[1, 0].plot(t2, waveform2)
axs[1, 0].set_title(file2)

t3 = np.linspace(0, len(waveform3) / fs3, num=len(waveform3))
axs[2, 0].plot(t3, waveform3)
axs[2, 0].set_title(file3)

# Plot spectrograms
f1, t_spec1, Sxx1 = signal.spectrogram(waveform1, fs1)
axs[0, 1].pcolormesh(1000*t_spec1, f1/1000, 10*np.log10(Sxx1/Sxx1.max()), vmin=-120, vmax=0, cmap='inferno')
axs[0, 1].set_title('Spectrogram')
axs[0, 1].set_ylabel('Frequency [Hz]')
axs[0, 1].set_xlabel('Time [sec]')

f2, t_spec2, Sxx2 = signal.spectrogram(waveform2, fs2)
axs[1, 1].pcolormesh(1000*t_spec2, f2/1000, 10*np.log10(Sxx2/Sxx2.max()), vmin=-120, vmax=0, cmap='inferno')
axs[1, 1].set_title('Spectrogram')
axs[1, 1].set_ylabel('Frequency [Hz]')
axs[1, 1].set_xlabel('Time [sec]')

f3, t_spec3, Sxx3 = signal.spectrogram(waveform3, fs3)
axs[2, 1].pcolormesh(1000*t_spec3, f3/1000, 10*np.log10(Sxx3/Sxx3.max()), vmin=-120, vmax=0, cmap='inferno')
axs[2, 1].set_title('Spectrogram')
axs[2, 1].set_ylabel('Frequency [Hz]')
axs[2, 1].set_xlabel('Time [sec]')

# Adjust layout
plt.tight_layout()
plt.show()

#### Determine average length of utterance sample files

In [24]:
def get_audio_lengths(directory):
    lengths = []
    for filename in os.listdir(directory):
        if filename.endswith(".mp3") or filename.endswith(".wav"):
            audio = AudioSegment.from_file(os.path.join(directory, filename))
            lengths.append(len(audio) / 1000.0)  # Convert length from milliseconds to seconds
    return lengths


def calculate_statistics(lengths):
    average_length = np.mean(lengths)
    std_deviation = np.std(lengths)
    return average_length, std_deviation


pre_lengths  = get_audio_lengths(pre_utterances)
post_lengths = get_audio_lengths(post_utterances)
dc_lengths   = get_audio_lengths(dc_utterances)

pre_avg_length, pre_std   = calculate_statistics(pre_lengths)
post_avg_length, post_std = calculate_statistics(post_lengths)
dc_avg_length, dc_std     = calculate_statistics(dc_lengths)

print(f"Average length of pre-anesthesia audio files: {pre_avg_length:.2f} +/- {pre_std:.2f} seconds")
print(f"Average length of post-anesthesia audio files: {post_avg_length:.2f} +/- {post_std:.2f} seconds")
print(f"Average length of at discharge audio files: {dc_avg_length:.2f} +/- {dc_std:.2f} seconds")

Average length of pre-anesthesia audio files: 0.84 +/- 0.13 seconds
Average length of post-anesthesia audio files: 0.93 +/- 0.19 seconds
Average length of at discharge audio files: 0.91 +/- 0.13 seconds


#### Determine shortest & longest length of utterance sample files

In [25]:
print(f"Pre-anesthesia audio files: Min = {min(pre_lengths)} seconds, Max = {max(pre_lengths)} seconds")
print(f"Post-anesthesia audio files: Min = {min(post_lengths)} seconds, Max = {max(post_lengths)} seconds")
print(f"Discharge audio files: Min = {min(dc_lengths)} seconds, Max = {max(dc_lengths)} seconds")

Pre-anesthesia audio files: Min = 0.51 seconds, Max = 1.32 seconds
Post-anesthesia audio files: Min = 0.54 seconds, Max = 1.59 seconds
Discharge audio files: Min = 0.63 seconds, Max = 1.29 seconds


### Step 4) Duration length adjustment

In [ ]:
# Example utterance spectrogram
demo_wav("pre/segment/01pre_nr_1.wav")

#### Convert all utterances to be of the same duration

In [27]:
def set_duration(file_path, target_duration_ms):
    audio = AudioSegment.from_file(file_path)
    if len(audio) > target_duration_ms:
        audio = audio[:target_duration_ms]
    else:
        silence = AudioSegment.silent(duration=target_duration_ms - len(audio))
        audio = audio + silence
    return audio


def process_directory(directory, target_duration_ms):
    for filename in os.listdir(directory):
        if filename.endswith(".wav"):
            file_path = os.path.join(directory, filename)
            new_audio = set_duration(file_path, target_duration_ms)
            new_audio.export(file_path, format="wav")


# Set duration limit to be the longest file
target_duration_ms = 1600
process_directory(pre_utterances, target_duration_ms)
process_directory(post_utterances, target_duration_ms)
process_directory(dc_utterances, target_duration_ms)

# Check audio dimensions again
pre_lengths  = get_audio_lengths(pre_utterances)
post_lengths = get_audio_lengths(post_utterances)
dc_lengths   = get_audio_lengths(dc_utterances)

pre_avg_length, pre_std   = calculate_statistics(pre_lengths)
post_avg_length, post_std = calculate_statistics(post_lengths)
dc_avg_length, dc_std     = calculate_statistics(dc_lengths)

print(f"Average length of pre-anesthesia audio files: {pre_avg_length:.2f} +/- {pre_std:.2f} seconds")
print(f"Average length of post-anesthesia audio files: {post_avg_length:.2f} +/- {post_std:.2f} seconds")
print(f"Average length of at discharge audio files: {dc_avg_length:.2f} +/- {dc_std:.2f} seconds")

Average length of pre-anesthesia audio files: 1.60 +/- 0.00 seconds
Average length of post-anesthesia audio files: 1.60 +/- 0.00 seconds
Average length of at discharge audio files: 1.60 +/- 0.00 seconds


In [ ]:
demo_wav("pre/segment/01pre_nr_1.wav")

### Step 5) Normalize audio files

Use ffmpeg-normalize (sudo ffmpeg-normalize *.wav -f -ext wav -ar 16000) on directory containing utterance files.

In [ ]:
# Show un-normalized utterance
demo_wav("dc/segment/02dc_nr_1.wav")

In [ ]:
# Compare ag normalized utterance
demo_wav("dc/segment/normalized/02dc_nr_1.wav")